<a href="https://colab.research.google.com/github/fellmaroua/BigData_tps/blob/Tp_BigData/Tp2BigData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"marouafellah","key":"500982e13d812ddf77223ad3716c2d3b"}'}

In [2]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [3]:
!kaggle datasets download -d dilwong/flightprices

Dataset URL: https://www.kaggle.com/datasets/dilwong/flightprices
License(s): Attribution 4.0 International (CC BY 4.0)
 99% 5.44G/5.51G [04:03<00:04, 18.4MB/s]
100% 5.51G/5.51G [04:03<00:00, 24.3MB/s]


In [4]:
import os

# المسار إلى الملف
path = "/content/flightprices.zip"

# حجم الملف بالبايت
size_bytes = os.path.getsize(path)

# تحويل إلى ميغابايت
size_mb = size_bytes / (1024 * 1024)
print(f" حجم الملف: {size_mb:.2f} ميغابايت")


 حجم الملف: 5646.49 ميغابايت


In [5]:
import os

def human_size(nbytes):
    for unit in ['B','KB','MB','GB','TB']:
        if nbytes < 1024:
            return f"{nbytes:.2f} {unit}"
        nbytes /= 1024
    return f"{nbytes:.2f} PB"

# عرض ملفات المجلد الحالي
for root, dirs, files in os.walk(".", topdown=True):
    # نعرض الملفات في الجذر فقط أولاً
    break

print("قائمة الملفات في المسار الحالي:")
for f in sorted(files):
    try:
        sz = os.path.getsize(f)
        print(f"  - {f} : {human_size(sz)}")
    except:
        print(f"  - {f}")

# نحسب حجم المجلد الكلي:
data_folder = "./"
total = 0
for dirpath, dirnames, filenames in os.walk(data_folder):
    for fname in filenames:
        fp = os.path.join(dirpath, fname)
        try:
            total += os.path.getsize(fp)
        except:
            pass

print(f"\n الحجم الكلي للمجلد /workspace: {human_size(total)}")


قائمة الملفات في المسار الحالي:
  - flightprices.zip : 5.51 GB

 الحجم الكلي للمجلد /workspace: 5.57 GB


In [6]:
import zipfile

zip_path = "flightprices.zip"
extract_dir = "./flightprices_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(" تم فك الضغط إلى:", extract_dir)
!ls -lh flightprices_data


 تم فك الضغط إلى: ./flightprices_data
total 29G
-rw-r--r-- 1 root root 29G Oct 22 18:27 itineraries.csv


In [7]:
import pandas as pd
import time
import os
import psutil

path = "/content/flightprices_data/itineraries.csv"

# الحصول على العملية الحالية لقياس الذاكرة
process = psutil.Process(os.getpid())

start_time = time.time()

chunk_size = 100000
total_mean = 0
count = 0

#  قبل البدء، نحسب الذاكرة الابتدائية
mem_before = process.memory_info().rss / (1024 ** 2)

for chunk in pd.read_csv(path, chunksize=chunk_size):
    if 'totalFare' not in chunk.columns:
        continue

    chunk['totalFare'] = pd.to_numeric(chunk['totalFare'], errors='coerce')
    chunk = chunk.dropna(subset=['totalFare'])
    total_mean += chunk['totalFare'].sum()
    count += chunk['totalFare'].count()

mean_price = total_mean / count

end_time = time.time()

#  بعد الانتهاء، نحسب الذاكرة النهائية
mem_after = process.memory_info().rss / (1024 ** 2)
mem_used = mem_after - mem_before

print(f" Pandas with chunks completed in {end_time - start_time:.2f} seconds")
print(f" Mean totalFare: {mean_price:.2f}")
print(f" Memory used: {mem_used:.2f} MB")


 Pandas with chunks completed in 897.37 seconds
 Mean totalFare: 340.39
 Memory used: 76.11 MB


In [8]:
import dask.dataframe as dd
import time
import psutil
import os

path = "/content/flightprices_data/itineraries.csv"

#  تهيئة قياس الذاكرة
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)

#  بدء توقيت التنفيذ
start = time.time()

# قراءة البيانات باستخدام Dask
df = dd.read_csv(path, assume_missing=True)
df['totalFare'] = dd.to_numeric(df['totalFare'], errors='coerce')
df = df.dropna(subset=['totalFare'])

# تنفيذ الحساب فعليًا (compute)
mean_price = df['totalFare'].mean().compute()

#  نهاية الوقت والذاكرة
end = time.time()
mem_after = process.memory_info().rss / (1024 ** 2)
mem_used = mem_after - mem_before

#  عرض النتائج
print(f" Dask completed in {end - start:.2f} seconds")
print(f" Mean totalFare: {mean_price:.2f}")
print(f" Memory used: {mem_used:.2f} MB")



 Dask completed in 852.81 seconds
 Mean totalFare: 340.39
 Memory used: 386.39 MB


In [9]:
import gzip
import shutil
import time
import os
import psutil

input_path = "/content/flightprices_data/itineraries.csv"
output_path = "/content/flightprices_data/itineraries.gz"

#  حساب الحجم الأصلي
size_before = os.path.getsize(input_path) / (1024 * 1024)
print(f" Original size: {size_before:.2f} MB")

#  بدء مراقبة الذاكرة والزمن
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)
start = time.time()

#  عملية الضغط
with open(input_path, 'rb') as f_in:
    with gzip.open(output_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

#  نهاية الزمن وحساب الذاكرة
end = time.time()
mem_after = process.memory_info().rss / (1024 ** 2)
mem_used = mem_after - mem_before

#  حساب الحجم الجديد ونسبة الضغط
size_after = os.path.getsize(output_path) / (1024 * 1024)
ratio = (1 - size_after / size_before) * 100

#  عرض النتائج النهائية
print(f" Compression done in {end - start:.2f} s")
print(f" New size: {size_after:.2f} MB ({ratio:.1f}% smaller)")
print(f" Memory used: {mem_used:.2f} MB")


 Original size: 29651.48 MB
 Compression done in 2231.48 s
 New size: 5372.84 MB (81.9% smaller)
 Memory used: -2.18 MB


In [ ]:
import gzip
import shutil
import time
import os
import psutil

input_path = "/content/flightprices_data/itineraries.csv"
output_path = "/content/flightprices_data/itineraries.gz"

#  حساب الحجم الأصلي
size_before = os.path.getsize(input_path) / (1024 * 1024)
print(f" Original size: {size_before:.2f} MB")

#  بدء مراقبة الذاكرة والزمن
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)
start = time.time()

#  عملية الضغط
with open(input_path, 'rb') as f_in:
    with gzip.open(output_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

#  نهاية الزمن وحساب الذاكرة
end = time.time()
mem_after = process.memory_info().rss / (1024 ** 2)
mem_used = mem_after - mem_before

#  حساب الحجم الجديد ونسبة الضغط
size_after = os.path.getsize(output_path) / (1024 * 1024)
ratio = (1 - size_after / size_before) * 100

#  عرض النتائج النهائية
print(f" Compression done in {end - start:.2f} s")
print(f" New size: {size_after:.2f} MB ({ratio:.1f}% smaller)")
print(f" Memory used: {mem_used:.2f} MB")


 Original size: 29651.48 MB
 Compression done in 2231.48 s
 New size: 5372.84 MB (81.9% smaller)
 Memory used: -2.18 MB


In [11]:
import pandas as pd

#  القيم التجريبية التي حصلت عليها من تشغيل الأكواد الثلاثة
data = {
    "Method": [
        "Pandas (chunk size)",
        "Dask",
        "Compression (gzip)"
    ],
    "Execution Time (s)": [
        897.37,
        852.81,
        2231.48
    ],
    "Memory Used (MB)": [
        76.11,
        386.39,
        -2.18
    ]
}

comparison_df = pd.DataFrame(data)

print(" Comparison between methods (TP02 Results):\n")
display(comparison_df)


 Comparison between methods (TP02 Results):



,Method,Execution Time (s),Memory Used (MB)
0,Pandas (chunk size),897.37,76.11
1,Dask,852.81,386.39
2,Compression (gzip),2231.48,-2.18
